# experiment.ipynb — Single-Input Full Pipeline Run

Runs the **complete** User-Adaptive XAI pipeline for **one input** in a single notebook:

| Stage | What it does |
|-------|--------------|
| 1. LIME | Runs LIME on the input text → top-6 features |
| 2. Ontology | Maps features to ontology ancestors (user-adaptive) |
| 3. LLM | Generates a readability-constrained explanation |
| 4. Analysis | Computes readability & faithfulness metrics |

**Only the final metrics CSV is saved** (to `outputs/exp_<EXPERIMENT_TAG>.csv`).

> **How to use:**
> 1. Edit the `EXPERIMENT RUN CONFIG` section in `config.py`.
> 2. Run this notebook top-to-bottom (Kernel → Restart & Run All).
> 3. Inspect the results table printed in the last cell.

## 0. Path bootstrap

In [1]:
import sys
from pathlib import Path

# Ensure this directory is on the path so all local modules are importable.
NOTEBOOK_DIR = Path(r"C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/MCC_contrained_decoding")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Working dir : {NOTEBOOK_DIR}")

Working dir : C:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\MCC_contrained_decoding


## 1. Imports

In [2]:
import json
import warnings

import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer

from config import (
    # Experiment knobs
    ABLATION_MODE,
    EXPERIMENT_RESULTS_PATH,
    EXPERIMENT_TAG,
    INPUT_TEXT,
    USER_CATEGORY,
    # LIME
    CLASS_NAMES,
    LIME_NUM_FEATURES,
    LIME_NUM_SAMPLES,
    # LLM / decoding
    LAMBDA_MAP,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
    # Data
    TOP_LIME_FEATURES,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_classifier, load_llm, load_ner_pipeline, load_ontology_model
from pipeline_helpers import (
    SYSTEM_PROMPT,
    build_prompt,
    enrich_with_ontology,
    generate_explanation,
    lime_coverage,
    make_lime_predictor,
    merge_entities,
    ontology_hit_rate,
    predict_class,
    readability_metrics,
)

warnings.filterwarnings("ignore")
print("✅ Imports complete.")

✅ Imports complete.


## 2. Experiment configuration (read-only — edit config.py)

In [3]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  EXPERIMENT CONFIG")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Tag                  : {EXPERIMENT_TAG}")
print(f"  User category        : {USER_CATEGORY}")
print(f"  Ablation mode        : {ABLATION_MODE}")
print(f"  Constrained decoding : {USE_CONSTRAINED_DECODING}")
print(f"  Lambda value         : {LAMBDA_MAP.get(USER_CATEGORY, 'N/A')}")
print(f"  Num beams            : {NUM_BEAMS}")
print(f"  Output path          : {EXPERIMENT_RESULTS_PATH}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXPERIMENT CONFIG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tag                  : expert_normal
  User category        : EXPERT
  Ablation mode        : normal
  Constrained decoding : True
  Lambda value         : 0.05
  Num beams            : 4
  Output path          : exp_output\exp_expert_normal.csv
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 3. Input text

In [4]:
if INPUT_TEXT is not None:
    input_text = INPUT_TEXT.strip()
    print("[Input] Using text from config.py INPUT_TEXT.")
else:
    data_path = NOTEBOOK_DIR / "test_data.txt"
    with open(data_path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    input_text = lines[0]
    print(f"[Input] INPUT_TEXT is None — using first line of test_data.txt.")

print(f"\nText ({len(input_text)} chars):")
print(input_text[:400] + ("…" if len(input_text) > 400 else ""))

[Input] Using text from config.py INPUT_TEXT.

Text (462 chars):
Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with endometriosis has been reported in rare cases, this patient was also noted to have massive destruction of the pelvic peritoneum. Failure of medical suppression necessitated total abdominal hysterectomy and bilateral salpingo-oophorectomy. Several months after surgery ascites res…


## 4. Load models (classifier, NER, ontology, LLM)

This is the slow step. All models are loaded once here and reused across stages.

In [5]:
# ── Classifier ────────────────────────────────────────────────────────────────
clf_model, clf_pipeline = load_classifier()

# ── NER (for multi-word entity merging in LIME) ───────────────────────────────
ner_pipeline = load_ner_pipeline()

# ── Ontology ──────────────────────────────────────────────────────────────────
ontology = load_ontology_model()

# ── LLM ───────────────────────────────────────────────────────────────────────
llm_tokenizer, llm_model = load_llm()

# ── Constrained-decoding generator ────────────────────────────────────────────
generator = None
if USE_CONSTRAINED_DECODING:
    generator = ReadabilityBeamGenerator(
        model=llm_model,
        tokenizer=llm_tokenizer,
        num_beams=NUM_BEAMS,
    )
    print(f"[CD] Constrained decoding enabled — λ={LAMBDA_MAP.get(USER_CATEGORY, '?')}, beams={NUM_BEAMS}")
else:
    print("[CD] Constrained decoding disabled (greedy/sample mode).")

print("\n✅ All models loaded.")

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Loader] Classifier ready.

[Loader] Loading NER model 'd4data/biomedical-ner-all' …


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[Loader] NER model ready.

[Ontology] Loading from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Ontology/doid.owl' …
[Ontology] Loaded successfully.
[Ontology] Stats → classes: 14493, est. max depth: 15
[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[Loader] LLM ready.

[CD] Constrained decoding enabled — λ=0.05, beams=4

✅ All models loaded.


## Stage 1 — LIME Feature Attribution

In [6]:
print("[Stage 1] Running NER-based entity merging …")
merged_text = merge_entities(input_text, ner_pipeline)

print("[Stage 1] Running LIME …")
lime_predictor = make_lime_predictor(clf_model, clf_pipeline)
explainer = LimeTextExplainer(class_names=CLASS_NAMES)

exp = explainer.explain_instance(
    merged_text,
    lime_predictor,
    num_features=LIME_NUM_FEATURES,
    num_samples=LIME_NUM_SAMPLES,
)

# Extract top features as [word, score] pairs
lime_features = exp.as_list()  # [(word, score), ...]

print(f"\n[Stage 1] Top {len(lime_features)} LIME features:")
for word, score in lime_features:
    print(f"  {word:25s}  score={score:+.4f}")

print("\n✅ Stage 1 complete.")

[Stage 1] Running NER-based entity merging …
[Stage 1] Running LIME …

[Stage 1] Top 6 LIME features:
  ascites                    score=+0.1900
  peritoneum                 score=+0.1639
  associated                 score=+0.0392
  Endometriosis              score=+0.0375
  with                       score=+0.0313
  resolved                   score=+0.0272

✅ Stage 1 complete.


## Stage 2 — Ontology Enrichment

In [7]:
print("[Stage 2] Classifying text …")
predicted_class, confidence = predict_class(input_text, clf_pipeline)
print(f"  Predicted class : {predicted_class}")
print(f"  Confidence      : {confidence:.4f}")

print("\n[Stage 2] Mapping LIME features to ontology ancestors …")
feature_data = enrich_with_ontology(
    lime_features=lime_features,
    ontology=ontology,
    user_category=USER_CATEGORY,
    ablation_mode=ABLATION_MODE,
)

hit_words = [f["feature_word"] for f in feature_data]
print(f"  Ontology hits : {hit_words} ({len(hit_words)}/{len(lime_features[:TOP_LIME_FEATURES])} features)")
for f in feature_data:
    print(f"    {f['feature_word']:20s} → {f['ancestors']}")

print("\n✅ Stage 2 complete.")

[Stage 2] Classifying text …
  Predicted class : Digestive system diseases
  Confidence      : 0.6624

[Stage 2] Mapping LIME features to ontology ancestors …
  Ontology hits : [np.str_('ascites'), np.str_('peritoneum'), np.str_('Endometriosis')] (3/6 features)
    ascites              → ['symptom', 'abdominal symptom', 'ascites']
    peritoneum           → ['multicellular anatomical structure', 'multi-tissue structure', 'serous membrane', 'peritoneum']
    Endometriosis        → ['disease of anatomical entity', 'reproductive system disease', 'female reproductive system disease', 'endometriosis']

✅ Stage 2 complete.


## Stage 3 — LLM Explanation Generation

In [8]:
print("[Stage 3] Building prompt …")
prompt = build_prompt(
    text=input_text,
    predicted_class=predicted_class,
    feature_data=feature_data,
    user_category=USER_CATEGORY,
)

if USE_CONSTRAINED_DECODING:
    lam = LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP["EXPERT"])
    print(f"[Stage 3] Generating with constrained decoding (λ={lam}, beams={NUM_BEAMS}) …")
else:
    print("[Stage 3] Generating with standard sampling …")

explanation = generate_explanation(
    text=input_text,
    predicted_class=predicted_class,
    feature_data=feature_data,
    user_category=USER_CATEGORY,
    tokenizer=llm_tokenizer,
    model=llm_model,
    generator=generator,
)

print("\n── Generated explanation ─────────────────────────────────────")
print(explanation)
print("─────────────────────────────────────────────────────────────")
print("\n✅ Stage 3 complete.")

[Stage 3] Building prompt …
[Stage 3] Generating with constrained decoding (λ=0.05, beams=4) …


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



── Generated explanation ─────────────────────────────────────
The model classified the abstract as a digestive system disease due to the presence of ascites, which is a symptom of the disease, and the mention of peritoneal destruction, indicating damage to the abdominal lining. These terms are directly relevant to diseases affecting the digestive system, such as inflammatory conditions or malignancies. Additionally, the reference to the failure of medical treatment and the need for surgical intervention further supports the classification, as surgical procedures are often associated with digestive system diseases.
─────────────────────────────────────────────────────────────

✅ Stage 3 complete.


## Stage 4 — Readability & Faithfulness Metrics

In [9]:
print("[Stage 4] Computing metrics …")

read_metrics = readability_metrics(explanation)
cov   = lime_coverage(explanation, feature_data)
hit   = ontology_hit_rate(feature_data)

result = {
    "experiment_tag":       EXPERIMENT_TAG,
    "user_category":        USER_CATEGORY,
    "ablation_mode":        ABLATION_MODE,
    "constrained_decoding": USE_CONSTRAINED_DECODING,
    "lambda":               LAMBDA_MAP.get(USER_CATEGORY, None),
    "predicted_class":      predicted_class,
    "confidence":           confidence,
    "text_snippet":         input_text[:120] + "…",
    "explanation":          explanation,
    "lime_coverage":        cov,
    "ontology_hit_rate":    hit,
    **read_metrics,
}

df = pd.DataFrame([result])

print("\n── Metrics ───────────────────────────────────────────────────")
metric_cols = [
    "user_category", "ablation_mode", "constrained_decoding",
    "predicted_class", "confidence",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate",
]
pd.set_option("display.max_colwidth", 40)
display(df[metric_cols])

print("\n✅ Stage 4 complete.")

[Stage 4] Computing metrics …

── Metrics ───────────────────────────────────────────────────


,user_category,ablation_mode,constrained_decoding,predicted_class,confidence,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,EXPERT,normal,True,Digestive system diseases,0.6624,12.318889,18.248642,19.287187,0.3333,1.0



✅ Stage 4 complete.


## 5. Save final results

In [10]:
EXPERIMENT_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPERIMENT_RESULTS_PATH, index=False)
print(f"✅ Results saved → '{EXPERIMENT_RESULTS_PATH}'")

print("\n── Full explanation ──────────────────────────────────────────")
print(f"  Class      : {predicted_class} (conf={confidence:.4f})")
print(f"  User       : {USER_CATEGORY}")
print(f"  Ablation   : {ABLATION_MODE}")
print(f"  Tag        : {EXPERIMENT_TAG}")
print("─────────────────────────────────────────────────────────────")
print(explanation)

✅ Results saved → 'exp_output\exp_expert_normal.csv'

── Full explanation ──────────────────────────────────────────
  Class      : Digestive system diseases (conf=0.6624)
  User       : EXPERT
  Ablation   : normal
  Tag        : expert_normal
─────────────────────────────────────────────────────────────
The model classified the abstract as a digestive system disease due to the presence of ascites, which is a symptom of the disease, and the mention of peritoneal destruction, indicating damage to the abdominal lining. These terms are directly relevant to diseases affecting the digestive system, such as inflammatory conditions or malignancies. Additionally, the reference to the failure of medical treatment and the need for surgical intervention further supports the classification, as surgical procedures are often associated with digestive system diseases.
